# 03 · Análise exploratória

**Objetivo:** entender os dados coletados antes de modelar. A pergunta principal: **o quanto a posição de largada já explica o resultado da corrida?** Esse vai ser o "baseline" que o modelo precisa superar.

## Setup

In [1]:
import pandas as pd 
import glob

## 1. Carregando os dados
Leio todos os CSVs salvos pelo notebook 02 e junto numa tabela só. Quando novas temporadas forem coletadas, basta rodar de novo.

In [5]:
arquivos = glob.glob("../data/dados_*.csv")
dados = pd.concat([pd.read_csv(arq) for arq in arquivos], ignore_index=True)

print(arquivos)
dados.shape


['../data\\dados_2018.csv', '../data\\dados_2019.csv', '../data\\dados_2020.csv', '../data\\dados_2021.csv', '../data\\dados_2022.csv', '../data\\dados_2023.csv', '../data\\dados_2024.csv', '../data\\dados_2026.csv']


(3353, 25)

## 2. Qualidade dos dados
Antes de analisar, verifico onde há valores vazios e se eles têm explicação. Vazios esperados (como Q2 e Q3 de quem foi eliminado) são mantidos; vazios sem explicação indicam falha na coleta.

In [6]:
# Quantidade de valores vazios por coluna
dados.isna().sum()

DriverNumber             0
BroadcastName         2475
Abbreviation             0
DriverId                20
TeamName                 0
TeamColor             2475
TeamId                  20
FirstName                0
LastName                 0
FullName                 0
HeadshotUrl           2502
CountryCode           2914
Position                22
ClassifiedPosition      20
GridPosition            22
Time                  1196
Status                  20
Points                  20
Laps                    20
QualiPosition           28
Q1                      65
Q2                     903
Q3                    1745
Ano                      0
GP                       0
dtype: int64

## 3. Limpeza das colunas
Removo as colunas que não ajudam a prever o resultado: as que vêm quase sempre vazias (a fonte não fornece) e as que só repetem a identificação do piloto.

In [10]:
colunas_removidas = [
    "BroadcastName", "TeamColor", "HeadshotUrl", "CountryCode", "FirstName", "LastName", "FullName", "DriverNumber"
]

dados = dados.drop(columns=colunas_removidas)
dados.columns

Index(['Abbreviation', 'DriverId', 'TeamName', 'TeamId', 'Position',
       'ClassifiedPosition', 'GridPosition', 'Time', 'Status', 'Points',
       'Laps', 'QualiPosition', 'Q1', 'Q2', 'Q3', 'Ano', 'GP'],
      dtype='object')

### Dicionário de colunas

**Como funciona a classificação (Q1, Q2 e Q3):** no sábado, os pilotos disputam a classificação, que define a ordem de largada de domingo. Ela funciona como um mata-mata em três partes: no **Q1** todos correm e os mais lentos são eliminados; no **Q2** mais alguns são eliminados; no **Q3** os 10 mais rápidos disputam a pole position. Em cada parte, conta a melhor volta do piloto. Quem foi eliminado não tem tempo nas partes seguintes, por isso há valores vazios.

| Coluna | Significado | Quando é conhecida |
|---|---|---|
| `Abbreviation` | Sigla do piloto (ex.: NOR, VER) | Antes da corrida |
| `DriverId` | Identificador único do piloto (chave) | Antes da corrida |
| `TeamName` / `TeamId` | Equipe do piloto | Antes da corrida |
| `GridPosition` | Posição de largada | Antes da corrida |
| `QualiPosition` | Posição na classificação de sábado | Antes da corrida |
| `Q1`, `Q2`, `Q3` | Melhor volta em cada parte da classificação | Antes da corrida |
| `Position` | **Posição final na corrida (o que queremos prever)** | Depois da corrida |
| `ClassifiedPosition` | Posição oficial (`R` para quem abandonou) | Depois da corrida |
| `Status` | Como terminou (`Finished`, `Retired`...) | Depois da corrida |
| `Points` | Pontos ganhos | Depois da corrida |
| `Laps` | Voltas completadas | Depois da corrida |
| `Time` | Tempo de prova (ou diferença para o vencedor) | Depois da corrida |
| `Ano` / `GP` | Temporada e Grande Prêmio | Antes da corrida |

> Colunas conhecidas **depois** da corrida não podem ser usadas para prever o resultado: seria vazamento de dados.